# References Crawling: Statistics and Analysis

## 1. Scope and measurement definitions

This chapter evaluates the reference-crawling component of Milestone 1:
retrieving reference metadata from Semantic Scholar for the assigned arXiv
publications. It distinguishes lookup completion, successful reference
retrieval, identifier coverage, and metadata completeness so that these
outcomes are not reported as a single success rate.

The measurement definitions below follow the assignment's **Extracting
References**, **Requirement**, and **Report Requirements (Jupyter Book)**
sections. They define how the results will be calculated; they are not
observed crawl results.

### 1.1 Unit of analysis

- A **publication** is one base arXiv ID selected from `papers_index.json`.
  Its version list describes the associated revisions. Multiple versions do
  not multiply the publication count or the number of crawled references.
- A **reference entry** is one item in a completed publication's reference
  snapshot. Entries without an arXiv ID, without an S2 ID, and duplicate
  entries remain in the local corpus and in the stated denominators.
- A **publishable reference row** must have both the citing `arxiv_id` and
  `ref_arxiv_id`, as required by the final dataset schema. An entry count is
  not a count of unique cited papers or deduplicated citation edges.

The chapter describes a publication-level API snapshot. Linking that
snapshot to several versions does not establish that their historical
bibliographies are identical.


In [ ]:
import pandas as pd
from itables import init_notebook_mode, show

# Embed the table assets for HTML/Jupyter Book output.
# This cell does not load crawl data or call any crawling API.
init_notebook_mode(all_interactive=False, connected=False)


### 1.2 Interpreting publication outcomes

A response is counted as completed only after its schema and reference count
have been checked. A verified empty list is a valid retrieval result; an API
failure or an unfinished page sequence is not an empty bibliography.

The table is generated from the outcome definitions in the next code cell.
Use its search field or click a column heading to explore the definitions.


In [ ]:
outcomes = pd.DataFrame(
    [
        ("ok", "Yes", "Yes", "Positive count", "Verified non-empty snapshot"),
        ("empty_refs", "Yes", "Yes", "Zero", "Verified count = 0 and an empty list"),
        ("not_in_s2", "Yes", "No", "Excluded", "Valid lookup did not find a publication"),
        ("Pending / invalid / unreadable / refresh pending", "No", "No", "Excluded",
         "No completed result for the current task; an older snapshot may still exist"),
    ],
    columns=["Outcome", "Lookup completed", "References retrieved",
             "Contribution to references-per-paper mean", "Interpretation"],
)
show(outcomes, caption="Publication outcomes and their statistical treatment",
     index=False, pageLength=5, scrollX=True, order=[])


### 1.3 Metrics and denominators

Let **N** be the number of selected base IDs, **R** the number with verified
reference retrieval (`ok` + `empty_refs`), and **C** the number of completed
lookups (`ok` + `empty_refs` + `not_in_s2`). Reference-entry metrics use only
completed retrieval snapshots from the same selected set.

S2-ID resolution is an observable proxy for reference identification, not
proof that every metadata field is present. We therefore report it alongside
field completeness. An unweighted mean of per-paper rates and a pooled rate
have different denominators and must retain different labels.

A zero denominator produces a missing value, displayed as **N/A**, rather
than an invented 0% or 100%. Reference means include verified empty papers
as zeros but exclude `not_in_s2` and unfinished tasks.


In [ ]:
metric_definitions = pd.DataFrame(
    [
        ("Selected publications", "Count of selected base IDs", "Publication", "papers_index.json",
         "Do not use the sum of version counts"),
        ("Successful reference retrievals", "R = ok + empty_refs", "Publication", "Validated per-paper outcomes",
         "not_in_s2 is reported separately"),
        ("Lookup completion rate", "C / N", "Fraction", "Validated per-paper outcomes + selected IDs",
         "Includes not_in_s2 as a completed lookup"),
        ("Reference retrieval success rate", "R / N", "Fraction", "Validated per-paper outcomes + selected IDs",
         "Does not treat lookup-not-found as reference retrieval"),
        ("Mean references per retrieved publication", "Sum(n_refs) / R", "Entry / publication", "Completed reference snapshots",
         "Includes empty_refs as zero; entries are not deduplicated"),
        ("Mean per-paper S2-ID resolution rate", "Mean(resolved entries / n_refs) over papers with n_refs > 0", "Fraction",
         "Per-paper reference counts", "Each non-empty paper has equal weight; an identification proxy"),
        ("Pooled S2-ID resolution rate", "Sum(resolved entries) / Sum(n_refs)", "Fraction", "Completed reference snapshots",
         "Each reference entry has equal weight; not full metadata completeness"),
        ("arXiv ID coverage", "Entries with ref_arxiv_id / all reference entries", "Fraction", "Completed reference snapshots",
         "Missing arXiv IDs remain in the denominator"),
        ("Eligible final references rows", "Entries with non-null citing and referenced arXiv IDs", "Row", "Completed reference snapshots",
         "Preserves one row per crawled entry; not a unique-edge count"),
        ("Metadata field completeness", "Non-missing values in a named field / reference entries in its stated scope", "Fraction per field",
         "Reference-level metadata", "Report title/authors/date/identifiers separately; state all-entry or eligible-row scope"),
        ("Module wall time", "Elapsed wall time of one explicitly selected crawl run", "Second", "One run JSON and its run context",
         "Includes waits/retries within the recorded measurement window; not end-to-end system time"),
        ("Average module wall time per attempted publication", "Module wall time / attempted publications", "Second / publication",
         "Same run JSON", "Report reused/newly completed counts and run mode; do not mix resume with fresh benchmarks"),
        ("Single-publication runtime", "Wall time of a dedicated one-publication benchmark", "Second", "One-publication timing evidence",
         "Cannot recover individual paper latency from a shared batch average"),
        ("Sampled peak and mean RAM", "Maximum and arithmetic mean of valid RSS samples", "MB", "Run memory samples / run JSON",
         "State sampling interval and process scope; unavailable measurements remain N/A"),
        ("Peak and final disk storage", "Measured peak; final bytes in an explicitly stated artifact scope", "Byte", "Disk measurement evidence",
         "Final size and net growth are not peak usage or physical bytes written"),
    ],
    columns=["Metric", "Calculation", "Unit", "Evidence", "Interpretation limit"],
)
show(metric_definitions, caption="Reference statistics and performance measurement definitions",
     index=False, pageLength=5, scrollX=True, order=[])


### 1.4 Evidence used by the report

The following inventory identifies the expected crawler artifacts and their
roles. It does not claim that these files have already been loaded or that
their contents have passed validation. The data-loading section will check
the selected IDs, schemas, snapshot completeness, and consistency between
per-paper data and summaries before publishing numeric results.


In [ ]:
evidence_inventory = pd.DataFrame(
    [
        ("papers_index.json", "Input publication IDs and associated versions", "Defines the selected population and N",
         "Use base IDs as jobs; retain version metadata separately"),
        ("references.json per publication", "Reference entries, status, counts, verification and refresh flag",
         "Primary evidence for snapshot validity and field completeness", "Do not accept an unfinished refresh as completed"),
        ("papers_stats.csv", "One summary row per selected publication", "Convenient publication-level analysis",
         "Cross-check its ID set and counts; a CSV alone does not establish metadata completeness"),
        ("summary_references.json", "Aggregated outcomes and entry coverage", "Cross-check recomputed statistics",
         "Check scope and freshness; do not count output outside the input set"),
        ("run_*.json", "Run context, HTTP counts, timing, sampled RAM and storage measurements",
         "Performance of one selected run", "Do not sum historical runs into a fresh benchmark or infer missing measurements"),
    ],
    columns=["Artifact", "Contents", "Role in this chapter", "Required check"],
)
show(evidence_inventory, caption="Evidence inventory for the reference-crawling report",
     index=False, pageLength=5, scrollX=True, order=[])


### 1.5 Reproducibility and interpretation boundaries

These tables are created by Python cells during compilation and rendered
as searchable, sortable HTML tables with ITables. This notebook requests
`execution_mode: force`, so its cells are executed even if saved outputs
exist. Compiling the report will not re-run the crawler.

The assignment's final performance comparison requires an **end-to-end,
CPU-only Google Colab benchmark**. Reference-module measurements will be
labelled by their actual machine, run context and process scope; they will
not be presented as that full-system benchmark. Source sizes before and
after excluding figures belong to the source-download analysis, not to
reference JSON statistics. Missing peak-disk or per-paper timing evidence
will remain unreported rather than being inferred from final size or batch
averages.

`ref_submission_date` currently carries the publication date returned by S2;
it has not been independently verified as the arXiv submission date. Missing
metadata and this provenance limit must accompany the eventual results.

**Technical references:** [ITables](https://github.com/mwouts/itables),
[Jupyter Book notebook execution](https://jupyter-book.readthedocs.io/v1/content/execute.html),
and [MyST-NB file-level execution settings](https://myst-nb.readthedocs.io/en/latest/configuration.html#file-level-configuration).
